# Math Assistant Agent — fine-tuning a Qwen3 math tutor

End-to-end pipeline, in the order it actually runs:

1. **Environment** — inline dependency installs (there is no `requirements.txt`).
2. **Data acquisition** — accepted, highly-voted Q/A pairs from the StackExchange API.
3. **Cleaning** — strip HTML while preserving the LaTeX/MathJax inside it.
4. **Dataset formatting** — ShareGPT-style `messages` JSONL for TRL's `SFTTrainer`.
5. **Model setup** — `Qwen3-4B-Thinking-2507` loaded in 4-bit, LoRA config on top (the "QLoRA" part).
6. **Fine-tuning** — train the adapter and save it (adapter only, ~MBs, not the 4B base).
7. **Merge & inference** — reattach the adapter to the base model and ask it a question.

> Cells must be run in order — later ones depend on names bound earlier
> (`meu_dataset`, `model`, `tokenizer`, `lora_config`, `bnb_config`).

---
## 1. Environment

Dependencies are installed inline. Restart the kernel after these if the imports below fail.

In [1]:
!pip install transformers peft bitsandbytes accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 32.2 MB/s eta 0:00:00:00:0100:01


In [2]:
!pip install beautifulsoup4

In [3]:
!pip install trl datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 863.2/863.2 kB 33.4 MB/s eta 0:00:00


---
## 2. Data acquisition

Training data comes from StackExchange. We start with a bare smoke test against the public
API — no key, no filters — just to confirm connectivity and see the shape of a response
before writing anything that depends on it.

In [8]:
import requests
import json

# URL base
url = "https://api.stackexchange.com/2.3/questions"

# Parâmetros exatos que você quer testar
params = {
    "order": "desc",
    "sort": "activity",
    "site": "stackoverflow"
}

print("Iniciando o teste da chamada à API...")

try:
    response = requests.get(url, params=params)
    
    # Verifica se a requisição foi um sucesso (HTTP Status 200)
    if response.status_code == 200:
        data = response.json()
        items = data.get("items", [])
        
        print("\n✅ Sucesso! A API respondeu perfeitamente.")
        print(f"Total de questões retornadas nesta página: {len(items)}\n")
        
        if items:
            # Extraímos os dados da primeira pergunta para validar a estrutura
            primeira = items[0]
            print("--- Exemplo do primeiro retorno ---")
            print(f"ID da Pergunta: {primeira.get('question_id')}")
            print(f"Título: {primeira.get('title')}")
            print(f"Foi respondida?: {primeira.get('is_answered')}")
            print(f"Link: {primeira.get('link')}")
            
    else:
        print(f"\n❌ Erro HTTP {response.status_code} ao contatar a API.")
        print("Detalhes do erro:", json.dumps(response.json(), indent=4))

except Exception as e:
    print(f"\n❌ Erro crítico de conexão (Network/Python): {e}")

Iniciando o teste da chamada à API...

✅ Sucesso! A API respondeu perfeitamente.
Total de questões retornadas nesta página: 30

--- Exemplo do primeiro retorno ---
ID da Pergunta: 79747999
Título: Command START_REPLICATION failing while trying to conusme logical replication
Foi respondida?: True
Link: https://stackoverflow.com/questions/79747999/command-start-replication-failing-while-trying-to-conusme-logical-replication


Now the real extraction. Two things make this dataset usable for supervised fine-tuning:

- `"accepted": "true"` + `"sort": "votes"` — we only want questions with a community-blessed
  answer, ranked by quality. A wrong answer teaches the model a wrong answer.
- the custom `filter` — StackExchange's default response omits answer bodies entirely; this
  filter pulls them in so a question and its accepted answer arrive in a single call.

The result is a list of `{question_id, title, prompt, completion}` dicts, still raw HTML.

In [9]:
import requests
import json

def fetch_math_dataset(api_key=None, num_questions=5):
    # Mudamos para search/advanced para suportar o filtro de 'accepted'
    url = "https://api.stackexchange.com/2.3/search/advanced"
    
    params = {
            "site": "mathoverflow.net",  # Mantive o MathOverflow, altere para "math" se quiser matemática geral
            "order": "desc",
            "sort": "votes",             # Voltando para 'votes' para priorizar a qualidade
            "accepted": "true",          # O filtro de status vital para o Fine-Tuning
            "pagesize": num_questions,
            "filter": "!6WPIomnMNcVD9"   # Seu novo filtro mágico gerado com sucesso!
        }
    
    if api_key:
        params["key"] = api_key

    print(f"Buscando {num_questions} questões estruturadas do Math Stack Exchange...")
    response = requests.get(url, params=params)
    
    if response.status_code != 200:
        print("❌ Erro na API:", response.json())
        return []

    data = response.json()
    dataset = []
    print(f"Response.json: {data}")
    
    # Estruturando os pares Prompt/Completion
    for item in data.get("items", []):
        prompt_text = item.get("body", "")
        completion_text = ""
        accepted_id = item.get("accepted_answer_id")
        
        # Busca a resposta correta dentro do array de respostas
        if "answers" in item:
            for ans in item["answers"]:
                if ans.get("answer_id") == accepted_id:
                    completion_text = ans.get("body", "")
                    break
        
        if prompt_text and completion_text:
            dataset.append({
                "question_id": item.get("question_id"),
                "title": item.get("title"),
                "prompt": prompt_text,
                "completion": completion_text
            })

    return dataset

# --- Testando a Extração ---
# Você pode passar sua api_key real como parâmetro: fetch_math_dataset(api_key="SUA_CHAVE")
meu_dataset = fetch_math_dataset(num_questions=3)

if meu_dataset:
    print("\n✅ Extração Estruturada Concluída!")
    print(f"Total de pares Prompt/Completion válidos: {len(meu_dataset)}\n")
    
    # Exibindo um pedaço do primeiro item para validar
    primeiro_item = meu_dataset[0]
    print("--- Exemplo de Estrutura do Modelo ---")
    print(f"Título: {primeiro_item['title']}")
    print(f"Prompt (Início): {primeiro_item['prompt'][:150]}...")
    print(f"Completion (Início): {primeiro_item['completion'][:150]}...")

Buscando 3 questões estruturadas do Math Stack Exchange...
Response.json: {'items': [{'tags': ['pr.probability', 'polynomials', 'cv.complex-variables'], 'answers': [{'owner': {'account_id': 1790716, 'reputation': 98147, 'user_id': 11142, 'user_type': 'registered', 'accept_rate': 48, 'profile_image': 'https://www.gravatar.com/avatar/b2b87c8281161bba80ee253b820a850f?s=256&d=identicon&r=PG', 'display_name': 'Igor Rivin', 'link': 'https://mathoverflow.net/users/11142/igor-rivin'}, 'is_accepted': False, 'score': 125, 'last_activity_date': 1672317724, 'last_edit_date': 1672317724, 'creation_date': 1412283861, 'answer_id': 182414, 'question_id': 182412, 'content_license': 'CC BY-SA 4.0', 'body': '<p>A complete derivation can be found in the classical paper of Shepp and Vanderbei:</p>\n<blockquote>\n<p>Larry A. Shepp and Robert J. Vanderbei: <a href="https://www.ams.org/journals/tran/1995-347-11/S0002-9947-1995-1308023-8/" rel="noreferrer">The complex zeros of random polynomials</a>, Trans. Am

---
## 3. Cleaning

The bodies are HTML. Stripping it naively would also eat the mathematics: StackExchange wraps
formulas in `<span class="math-container">$...$</span>`, and the `$...$` LaTeX inside is exactly
what we want to keep. BeautifulSoup's `get_text()` unwraps the tag while leaving that content
intact — **this is the load-bearing invariant of this step**; don't replace it with a regex
that treats `$` as markup.

Paragraph and list tags are turned into newlines/bullets first, so the flattened text keeps its
structure instead of collapsing into one line.

In [10]:
import json
import re
from bs4 import BeautifulSoup

# Simulação do retorno da sua API
dados_brutos = [
    {
        'question_id': 182412,
        'title': 'Why do roots of polynomials tend to have absolute value close to 1?',
        'prompt': '<p>While playing around with Mathematica I noticed...</p>', # (truncado para o exemplo)
        'completion': '<p>Let me give an informal explanation...</p>'
    },
    # ... seus outros dados
]

def clean_html_for_math(html_text):
    """
    Remove tags HTML, converte parágrafos em quebras de linha e 
    preserva o conteúdo das fórmulas (MathJax/LaTeX).
    """
    if not html_text:
        return ""
        
    soup = BeautifulSoup(html_text, "html.parser")
    
    # Adiciona quebras de linha duplas após parágrafos, blocos de código e títulos
    for tag in soup.find_all(['p', 'br', 'h1', 'h2', 'h3', 'pre', 'div']):
        tag.insert_after('\n\n')
        
    # Transforma tags de lista em bullet points
    for li in soup.find_all('li'):
        li.insert_before('- ')
        li.insert_after('\n')
        
    # Extrai todo o texto (O BeautifulSoup inteligentemente extrai o conteúdo
    # $...$ de dentro da tag <span class="math-container">)
    texto_limpo = soup.get_text()
    
    # Remove quebras de linha excessivas (mais de 2 vira apenas 2)
    texto_limpo = re.sub(r'\n{3,}', '\n\n', texto_limpo)
    
    return texto_limpo.strip()

---
## 4. Dataset formatting

`SFTTrainer` and the Qwen3 chat template both expect ShareGPT-style conversations, so each
cleaned pair becomes `{"messages": [system, user, assistant]}`:

- **system** — fixes the persona (step-by-step, rigorous) that we later reuse verbatim at
  inference time. Training and inference prompts must match or the tuning doesn't transfer.
- **user** — title *and* body. The title is often the sharpest statement of the problem.
- **assistant** — the accepted answer, the target the model learns to produce.

Written out as JSONL (one conversation per line) to `dataset_math_qlora.jsonl`.

In [11]:
def prepare_dataset(dados):
    """
    Converte os dados limpos para o formato de mensagens esperado pelo Qwen3.
    """
    dataset_formatado = []
    
    # System prompt que define a persona do seu agente
    system_prompt = (
        "Você é um assistente de matemática avançado. Resolva os problemas "
        "passo a passo, fornecendo explicações claras e rigorosas."
    )
    
    for item in dados:
        # 1. Limpar os textos
        pergunta_limpa = clean_html_for_math(item['prompt'])
        resposta_limpa = clean_html_for_math(item['completion'])
        
        # 2. Juntar o título com o corpo da pergunta para dar mais contexto ao modelo
        conteudo_usuario = f"Título: {item['title']}\n\n{pergunta_limpa}"
        
        # 3. Montar a estrutura de mensagens
        conversacao = {
            "messages": [
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": conteudo_usuario},
                {"role": "assistant", "content": resposta_limpa}
            ]
        }
        
        dataset_formatado.append(conversacao)
        
    return dataset_formatado

# --- Execução ---
meu_dataset_pronto = prepare_dataset(meu_dataset)

# Salvando no formato JSONL (JSON Lines) - Ideal para HuggingFace
caminho_arquivo = "dataset_math_qlora.jsonl"
with open(caminho_arquivo, "w", encoding="utf-8") as f:
    for registro in meu_dataset_pronto:
        f.write(json.dumps(registro, ensure_ascii=False) + "\n")

print(f"✅ Dataset estruturado e salvo em: {caminho_arquivo}")

# Exibindo como ficou o primeiro item formatado
print("\n--- Estrutura Final do Primeiro Item ---")
print(json.dumps(meu_dataset_pronto[0], indent=2, ensure_ascii=False))

✅ Dataset estruturado e salvo em: dataset_math_qlora.jsonl

--- Estrutura Final do Primeiro Item ---
{
  "messages": [
    {
      "role": "system",
      "content": "Você é um assistente de matemática avançado. Resolva os problemas passo a passo, fornecendo explicações claras e rigorosas."
    },
    {
      "role": "user",
      "content": "Título: Why do roots of polynomials tend to have absolute value close to 1?\n\nWhile playing around with Mathematica I noticed that most polynomials with real coefficients seem to have most complex zeroes very near the unit circle. For instance, if we plot all the roots of a polynomial of degree 300 with coefficients chosen randomly from the interval $[27, 42]$, we get something like this:\n\nThe Mathematica code to produce the picture was:\n\nrandomPoly[n_, x_, {a_, b_}] := \n  x^Range[0, n] . Table[RandomReal[{a, b}], {n + 1}];\nGraphics[Point[{Re[x], Im[x]}] /. \n  NSolve[randomPoly[300, x, {27, 42}], x], Axes -> True]\n\nIf I try other interva

---
## 5. Model setup — the QLoRA stack

Three layers stack up here, and the order matters:

1. **4-bit quantization** (`BitsAndBytesConfig`) — the *Q*. NF4 + double quantization brings
   a 4B model onto a single consumer GPU; compute still happens in `bfloat16`.
2. **The frozen base model** — loaded under that quantization config.
3. **LoRA adapters** (`LoraConfig`) — small trainable matrices on every attention and MLP
   projection. Only these get gradients.

⚠️ **Multi-GPU gotcha**: `CUDA_VISIBLE_DEVICES="0"` must be set *before* `torch` is imported,
and training must use `device_map={"": 0}`. Letting `device_map="auto"` shard across GPUs makes
`SFTTrainer` wrap the model in `DataParallel` and the run breaks. Inference can use `"auto"`.

In [4]:
import os
# Isso "esconde" qualquer outra GPU, fazendo o PyTorch achar que você só tem 1 placa.
# Deve ser executado ANTES de importar torch, transformers ou peft.
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

The quantization config itself. `bnb_4bit_compute_dtype=torch.bfloat16` is the dtype used for
the actual matmuls after dequantization — switch it to `float16` on GPUs older than Ampere.

In [5]:
model_name = "Qwen/Qwen3-4B-Thinking-2507"

# 1. Configuração do BitsAndBytes para quantização em 4-bits (O "Q" do QLoRA)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16 # Use float16 se sua GPU não suportar bfloat16
)

The tokenizer. Qwen3 ships no `pad_token`, and batching without one raises at collation time —
hence the fallback to `eos_token`.

In [6]:
# 2. Carregando o Tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name)
# É importante definir o token de preenchimento (pad_token) para o treinamento
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

Load the base model in 4-bit, then declare the LoRA config. The `target_modules` list covers
every attention (`q/k/v/o_proj`) and MLP (`gate/up/down_proj`) projection — adapting all of
them rather than attention alone measurably helps on reasoning-heavy tasks.

Note the config is only *declared* here; `SFTTrainer` applies it to the model in the next
section.

In [7]:
# 1. Carregue o modelo apenas com a quantização (Remova o get_peft_model e prepare_kbit_training)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map={"": 0}, # <--- A MUDANÇA ESTÁ AQUI    torch_dtype="auto"
    torch_dtype="auto"
)

# 2. Defina o LoraConfig normalmente
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

---
## 6. Fine-tuning

Everything is in place: a formatted dataset on disk, a quantized base model in memory, and a
LoRA config. Now train.

In [12]:
import transformers

print(transformers.__version__)

from datasets import load_dataset
from trl import SFTTrainer, SFTConfig
import torch

5.0.0


Read the JSONL back through `datasets`. TRL recognizes the `messages` column and applies the
Qwen3 chat template itself, so no manual token formatting is needed.

In [13]:
# 1. Carregando o dataset estruturado
# Usamos a biblioteca 'datasets' do Hugging Face para ler o JSONL eficientemente
dataset = load_dataset("json", data_files="dataset_math_qlora.jsonl", split="train")
print(dataset)

Generating train split: 0 examples [00:00, ? examples/s]

Dataset({
    features: ['messages'],
    num_rows: 3
})


The training run. Two settings deserve attention:

- `max_steps=1` — **this is a smoke test, not a real run.** It proves the plumbing works
  end-to-end. Swap it for `num_train_epochs` once you're happy with the pipeline.
- `gradient_accumulation_steps=4` with `per_device_train_batch_size=1` — an effective batch of
  4 without the VRAM cost of an actual batch of 4.

`peft_config` goes straight to `SFTTrainer`, which wraps the model in PEFT for us. Saving
writes **only the adapter** (a few MB), not the merged 4B model.

In [15]:
# 2. Configurações de Treinamento (SFTConfig)
training_args = SFTConfig(
    output_dir="./qwen-math-agent",
    per_device_train_batch_size=1,  # Reduzido para caber na memória (ajuste conforme sua GPU)
    gradient_accumulation_steps=4,  # Simula um batch_size maior sem usar mais VRAM
    learning_rate=2e-4,             # Taxa de aprendizado padrão para QLoRA
    logging_steps=10,
    max_steps=1,                  # Apenas para o teste inicial! Depois altere para num_train_epochs
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    optim="paged_adamw_8bit",       # Otimizador eficiente em memória
    dataset_text_field="messages",  # O TRL reconhece a estrutura de chat automaticamente
    max_length=2048,            # Limite de tokens para o treinamento
    packing=False,
)

# 3. Passe o peft_config diretamente para o SFTTrainer!
trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
    processing_class=tokenizer,
    peft_config=lora_config # <--- O TRL vai envelopar o modelo pra você aqui
)

# 4. Executando o Treinamento
print("Iniciando o Fine-Tuning do seu Agente Matemático...")
trainer.train()

# 5. Salvando os Adaptadores LoRA
# Isso salva apenas os pesos treinados (poucos MBs), não o modelo base inteiro de 4B.
trainer.model.save_pretrained("./qwen-math-agent-adapter")
tokenizer.save_pretrained("./qwen-math-agent-adapter")
print("Treinamento concluído e adaptadores salvos com sucesso!")

/usr/local/lib/python3.12/dist-packages/peft/mapping_func.py:72: UserWarning: You are trying to modify a model with PEFT for a second time. If you want to reload the model with a different config, make sure to call `.unload()` before.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:302: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


Iniciando o Fine-Tuning do seu Agente Matemático...


Step,Training Loss


Treinamento concluído e adaptadores salvos com sucesso!


Quick check that the adapter directory actually landed on disk.

In [16]:
os.listdir()

['qwen-math-agent',
 'qwen-math-agent-adapter',
 '.virtual_documents',
 'dataset_math_qlora.jsonl',
 'state.db']

---
## 7. Merge & inference

Fresh load of the base model, with the trained adapter attached on top via `PeftModel` — this
is what deployment looks like (later served behind vLLM/Ollama for the agent layer).

The `solver()` function below has one subtlety worth calling out. Qwen3-Thinking emits its
reasoning inside `<think>...</think>` before the answer, and we split the two by **locating
token id `151668`** (the `</think>` close tag) in the generated ids — not by regex on decoded
text, which is fragile. This matters beyond display: when persisting multi-turn history, only
the final-answer portion may be fed back, since raw `<think>` content in history degrades
subsequent reasoning.

Generation params (`temperature=0.6`, `top_p=0.95`, `top_k=20`) are the model card's official
recommendations — they are not arbitrary and shouldn't be tuned casually.

In [18]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

base_model_name = "Qwen/Qwen3-4B-Thinking-2507"
adapter_path = "./qwen-math-agent-adapter"

print("1. Configurando ambiente e carregando Tokenizer...")
# Configuração para carregar o modelo base leve em 4-bits
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16 # ou float16 se sua placa for mais antiga
)

# Carregamos o tokenizer que foi salvo junto com seu adaptador
tokenizer = AutoTokenizer.from_pretrained(adapter_path)

print("2. Carregando Modelo Base...")
base_model = AutoModelForCausalLM.from_pretrained(
    base_model_name,
    quantization_config=bnb_config,
    device_map="auto", # Agora podemos voltar para 'auto' para inferência!
    torch_dtype="auto"
)

print("3. Acoplando o seu Agente (Adaptador LoRA)...")
# Essa é a mágica: une o modelo base com o que ele aprendeu no StackExchange
model = PeftModel.from_pretrained(base_model, adapter_path)
model.eval() # Coloca o modelo explicitamente em modo de avaliação/inferência
print("✅ Modelo pronto para uso!")

# ---------------------------------------------------------
# Função de Inferência
# ---------------------------------------------------------
def solver(pergunta):
    # Usamos o mesmo system prompt do treinamento para ativar a "persona" correta
    messages = [
        {"role": "system", "content": "Você é um assistente de matemática avançado. Resolva os problemas passo a passo, fornecendo explicações claras e rigorosas."},
        {"role": "user", "content": pergunta}
    ]
    
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )
    
    model_inputs = tokenizer([text], return_tensors="pt").to(model.device)
    
    print("Pensando...\n")
    with torch.no_grad(): # Desativa cálculo de gradientes (economiza VRAM e CPU)
        generated_ids = model.generate(
            **model_inputs,
            max_new_tokens=4096, # Ajuste conforme sua VRAM (oficialmente recomendam mais alto para matemática)
            temperature=0.6,     # Recomendação oficial para o Qwen3
            top_p=0.95,
            top_k=20
        )
        
    # Isolar apenas a resposta gerada (tirando o prompt de entrada)
    output_ids = generated_ids[0][len(model_inputs.input_ids[0]):].tolist() 
    
    # Fazendo o parse do conteúdo de "Thinking" (A tag de fechamento é o token 151668)
    try:
        index = len(output_ids) - output_ids[::-1].index(151668)
        thinking_content = tokenizer.decode(output_ids[:index], skip_special_tokens=True).strip("\n")
        final_answer = tokenizer.decode(output_ids[index:], skip_special_tokens=True).strip("\n")
    except ValueError:
        thinking_content = "Não foi possível separar o raciocínio."
        final_answer = tokenizer.decode(output_ids, skip_special_tokens=True).strip("\n")
        
    return thinking_content, final_answer

# --- Testando o Modelo ---
minha_pergunta = "Qual é a derivada de f(x) = x^2 * sin(x) ? Mostre os passos."
pensamento, resposta_final = solver(minha_pergunta)

print("=== RACIOCÍNIO INTERNO (THINKING) ===")
print(pensamento)
print("\n=== RESPOSTA FINAL ===")
print(resposta_final)

1. Configurando ambiente e carregando Tokenizer...
2. Carregando Modelo Base...


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

3. Acoplando o seu Agente (Adaptador LoRA)...
✅ Modelo pronto para uso!
Pensando...

=== RACIOCÍNIO INTERNO (THINKING) ===
Okay, so I need to find the derivative of f(x) = x² * sin(x). Hmm, let me think. I remember from my calculus class that when you have a product of two functions, you use the product rule. Right, the product rule states that if you have two functions multiplied together, say u(x) and v(x), then the derivative is u’(x)v(x) + u(x)v’(x). 

Alright, so in this case, the two functions are x² and sin(x). Let me set u(x) = x² and v(x) = sin(x). Then I need to find u’(x) and v’(x) first.

Let me recall the derivatives of these functions. The derivative of x² with respect to x is 2x, so u’(x) = 2x. And the derivative of sin(x) is cos(x), so v’( x) = cos(x). 

Now applying the product rule: f’(x) = u’(x)v(x) + u(x)v’(x). Plugging in the values I found, that would be 2x * sin(x) + x² * cos(x). 

Wait, let me check if that's right. Let me make sure I didn't mix up anything. So 

A second probe, this time a proof question rather than a computation — checking the model
handles open-ended mathematical reasoning, not just mechanical differentiation.

In [19]:
# --- Testando o Modelo ---
minha_pergunta = "Como eu provo o teorema do sanduiche de analise?"
pensamento, resposta_final = solver(minha_pergunta)

print("=== RACIOCÍNIO INTERNO (THINKING) ===")
print(pensamento)
print("\n=== RESPOSTA FINAL ===")
print(resposta_final)

Pensando...

=== RACIOCÍNIO INTERNO (THINKING) ===
Okay, so I need to prove the Sandwich Theorem, also known as the Squeeze Theorem in analysis. Let me recall what the theorem says. From what I remember, if you have three functions f, g, and h, and they all converge to the same limit L as x approaches some point c, then the function g(x) is squeezed between f(x) and h(x) near c, so g(x) must also converge to L. Wait, no, maybe it's the other way around. Let me check.

Actually, the Squeeze Theorem states that if f(x) ≤ g(x) ≤ h(x) for all x in some neighborhood of c (except maybe at c itself), and if lim_{x→c} f(x) = lim_{x→c} h(x) = L, then lim_{x→c} g(x) = L. Yeah, that sounds right.

So, the problem is to prove this theorem. Let me try to work through it step by step.

First, I need to recall the precise statement of the theorem. Let me write it down formally:

Suppose that for all x in some interval (a, b) containing c (except possibly x = c), we have f(x) ≤ g(x) ≤ h(x). If lim_{x→

---
## Scratch